# Създаване на приложения за генериране на изображения (OpenAI)

Голямото езиково моделиране (LLMs) не се ограничава само до генериране на текст. Можете също да генерирате изображения от текстови описания. Изображенията като модалитет са полезни в МедТек, архитектурата, туризма, разработката на игри, маркетинга и други. В този урок работим с днешните **GPT Image** модели на платформата OpenAI.

## Цели на обучението

В края на този урок ще можете да:

- Обясните какво е генериране на изображения и къде е полезно.
- Разберете семейството от модели `gpt-image` и как се различава от наследените модели DALL·E.
- Създадете приложение за генериране на изображения и да редактирате изображения.

## Какво е генериране на изображения?

Моделите за генериране на изображения създават изображения от текстов прозорец. Модерни модели като `gpt-image` научават връзката между текста и изображенията по време на обучението и след това итеративно преобразуват случаен шум в изображение, което съответства на вашия прозорец.

Две добре познати семейства модели за изображения са:

- **`gpt-image` (OpenAI)** — настоящото поколение, използвано в този урок. Поддържа генериране на изображения от текст и редактиране на изображения (inpainting с маска).
- **Midjourney** — популярен външен модел със собствена услуга и работен процес, базиран на Discord.

> По-старите модели на OpenAI за изображения — **DALL·E 2** и **DALL·E 3** — са наследствени. DALL·E 3 вече не е достъпен за нови разгръщания, а функции като `create_variation` съществуваха само в DALL·E 2. Използвайте моделите `gpt-image` за нови приложения.

> **Важно:** Моделите `gpt-image` връщат генерираното изображение като **base64** (`b64_json`), а не като URL. Вашият код декодира низа base64 в байтове и го записва — няма URL на изображение за изтегляне.


## Изграждане на първото ви приложение за генериране на изображения

Какво е необходимо, за да създадете приложение за генериране на изображения? Трябват ви следните библиотеки:

- **python-dotenv**, силно се препоръчва да използвате тази библиотека, за да съхранявате тайните си в *.env* файл отделно от кода.
- **openai**, тази библиотека се използва за взаимодействие с OpenAI API.
- **pillow**, за работа с изображения в Python.


1. Създайте файл *.env* със следното съдържание:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. Съберете горните библиотеки в файл, наречен *requirements.txt*, по следния начин:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. След това създайте виртуална среда и инсталирайте библиотеките:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> За Windows използвайте следните команди, за да създадете и активирате вашата виртуална среда:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Добавете следния код във файл с име *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Нека обясним този код:

- Първо, импортираме необходимите библиотеки, включително библиотеката OpenAI, библиотеката dotenv, модула base64 и библиотеката Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- След това създаваме клиента, който чете API ключа от вашия ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Следващата стъпка е да генерираме изображението:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` моделите връщат изображението като **base64** низ в `data[0].b64_json`. Декодираме го в байтове и го записваме във файл — няма URL за изтегляне.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Накрая отваряме изображението и използваме стандартния визуализатор за изображения, за да го покажем:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Повече детайли за генерирането на изображението

Нека разгледаме параметрите на `images.generate`:

- **model** е моделът за изображения, който да се използва (например `gpt-image-2`).
- **prompt** е текстовото описание, използвано за генериране на изображението. Тук е "Зайче на кон, държащо близалка, на мъглива поляна, където растат нарциси".
- **size** е размерът на генерираното изображение (например `1024x1024`, `1536x1024`, `1024x1536` или `"auto"`).
- **n** е броят на генерираните изображения. Тук генерираме едно.

> Моделите за изображения не приемат параметър `temperature` — той е контрол за генериране на текст. За да получите разнообразие, извикайте API-то отново; за да намалите разнообразието, направете описанието си по-конкретно.

## Допълнителни възможности при генериране на изображения

Видяхте как да генерирате изображение с няколко реда Python. `gpt-image` моделите могат също да **редактират** съществуващо изображение. Като подадете изображение, опционална **маска** (която маркира областта за промяна) и описание, можете да промените част от изображение — например да добавите шапка на нашето зайче.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Основното изображение съдържа само заека; крайното изображение има шапката на заека.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
